In [ ]:
# Models
import os
from pprint import pprint

from langchain_ollama import ChatOllama, OllamaEmbeddings

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")

llm = ChatOllama(
    base_url=OLLAMA_HOST,
    keep_alive="12m",
    model="gemma4:e2b",
    temperature=0.24,
    reasoning=False,
)

embedding_model = OllamaEmbeddings(
    base_url=OLLAMA_HOST,
    model="nomic-embed-text",
)

pprint(llm)
pprint(embedding_model)

In [ ]:
# Search Sources
import os
from pprint import pprint
from typing import NotRequired, TypedDict

import trafilatura
from IPython.display import Image, display
from langchain_community.utilities import SearxSearchWrapper
from langgraph.graph import END, START, StateGraph

from src.prompts import search_sources_prompt_template
from src.subjects import TAROT_CARDS, Subject

SEARXNG_HOST = os.environ.get("SEARXNG_HOST", "http://localhost:8889")


class SubjectState(TypedDict):
    subject: Subject
    query: NotRequired[str]
    sources: NotRequired[str]


def search_sources(state: SubjectState) -> SubjectState:
    print("Searching sources...")
    subject = state.get("subject")
    query = search_sources_prompt_template.format(
        subject_name=subject.name,
        category_name=subject.category.name,
    )

    searxng_wrapper = SearxSearchWrapper(searx_host=SEARXNG_HOST)
    searxng_results = searxng_wrapper.results(query, num_results=12)

    trafilatura_results = []
    for i, searxng_result in enumerate(searxng_results, start=1):
        url = searxng_result.get("link", "")
        title = searxng_result.get("title", "").strip()
        text = trafilatura.extract(
            trafilatura.fetch_url(url),
            include_comments=False,
            include_tables=True,
            no_fallback=False,
        )
        if not text:
            print(f"Warning: No text extracted from URL: {url}")
            continue
        heading = title if title else url
        trafilatura_results.append(
            f"## Search Result {i}: {heading}\n\n**Source:** [{url}]({url})\n\n{text}"
        )

    if not trafilatura_results:
        raise ValueError(f"No content could be extracted for query: {query}")

    sources = "\n\n---\n\n".join(trafilatura_results)

    return {
        "subject": subject,
        "query": query,
        "sources": sources,
    }


graph_builder = StateGraph(SubjectState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", END)
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

state = graph.invoke({"subject": TAROT_CARDS[0]})
pprint(state)

In [ ]:
# Generate Semantic Document from Sources
from pprint import pprint
from typing import NotRequired

import inflect
from langchain_core.output_parsers import StrOutputParser

from src.output import write_semantics_document_markdown
from src.prompts import semantic_generate_document_from_sources_prompt_template

inflect_engine = inflect.engine()
str_output_parser = StrOutputParser()


class GenerateDocumentFromSourcesState(SubjectState):
    document: NotRequired[str]


def generate_semantic_document(
    state: GenerateDocumentFromSourcesState,
) -> dict[str, str]:
    print("Generating semantic document from sources...")
    subject = state["subject"]
    messages = semantic_generate_document_from_sources_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        category_name_plural=inflect_engine.plural(subject.category.name),  # type: ignore[arg-type]
        sources=state.get("sources"),
    )
    document = str_output_parser.invoke(llm.invoke(messages))
    write_semantics_document_markdown(subject, document)
    return {"document": document}


graph_builder = StateGraph(GenerateDocumentFromSourcesState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_node("generate_semantic_document", generate_semantic_document)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", "generate_semantic_document")
graph_builder.add_edge("generate_semantic_document", END)
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

state = graph.invoke({"subject": TAROT_CARDS[0]})
pprint(state)

In [ ]:
# Generate Semantic Document from Source Analysis
from typing import NotRequired

import inflect
from langchain_core.output_parsers import StrOutputParser

from src.prompts import (
    semantic_analyze_sources_prompt_template,
    semantic_generate_document_prompt_template,
)

inflect_engine = inflect.engine()
str_output_parser = StrOutputParser()


class GenerateDocumentFromAnalysisState(GenerateDocumentFromSourcesState):
    source_analysis: NotRequired[str]


def analyze_sources(state: GenerateDocumentFromAnalysisState) -> dict[str, str]:
    print("Analyzing sources...")
    subject = state["subject"]
    messages = semantic_analyze_sources_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        category_name_plural=inflect_engine.plural(subject.category.name),  # type: ignore[arg-type]
        sources=state.get("sources", ""),
    )
    source_analysis = str_output_parser.invoke(llm.invoke(messages))
    return {"source_analysis": source_analysis}


def generate_semantic_document(state: GenerateDocumentFromAnalysisState) -> dict[str, str]:
    print("Generating semantic document from source analysis...")
    subject = state["subject"]
    messages = semantic_generate_document_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        category_name_plural=inflect_engine.plural(subject.category.name),  # type: ignore[arg-type]
        source_analysis=state.get("source_analysis", ""),
    )
    document = str_output_parser.invoke(llm.invoke(messages))
    write_semantics_document_markdown(subject, document)
    return {"document": document}


graph_builder = StateGraph(GenerateDocumentFromAnalysisState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_node("analyze_sources", analyze_sources)
graph_builder.add_node("generate_semantic_document", generate_semantic_document)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", "analyze_sources")
graph_builder.add_edge("analyze_sources", "generate_semantic_document")
graph_builder.add_edge("generate_semantic_document", END)
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

state = graph.invoke({"subject": TAROT_CARDS[0]})
pprint(state)

In [ ]:
# Semantic Embeddings
from pathlib import Path

import numpy as np

SEMANTICS_OUTPUT_DIR = Path("../../output/semantics")

subject_slugs: list[str] = []
vectors: list[list[float]] = []

for category_dir in sorted(SEMANTICS_OUTPUT_DIR.iterdir()):
    if not category_dir.is_dir():
        continue
    for doc_path in sorted(category_dir.glob("*.md")):
        slug = doc_path.stem
        text = doc_path.read_text(encoding="utf-8")
        vector = embedding_model.embed_query(text)
        subject_slugs.append(f"{category_dir.name}/{slug}")
        vectors.append(vector)
        print(f"Embedded: {category_dir.name}/{slug}")

embedding_matrix = np.array(vectors, dtype=np.float32)
print(f"\nEmbedding matrix shape: {embedding_matrix.shape}")
print(f"Subjects: {len(subject_slugs)}")

In [ ]:
# Semantic Vector Math
import numpy as np


def vec(slug: str) -> np.ndarray:
    """Return the embedding vector for a subject by its slug (category/order-name)."""
    idx = next(i for i, s in enumerate(subject_slugs) if slug in s)
    return embedding_matrix[idx]


def nearest(
    query_vec: np.ndarray, n: int = 5, exclude: list[str] | None = None
) -> list[tuple[str, float]]:
    """Return the n nearest subjects to a query vector by cosine similarity."""
    exclude = exclude or []
    norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
    normed = embedding_matrix / np.where(norms == 0, 1, norms)
    query_normed = query_vec / (np.linalg.norm(query_vec) or 1)
    scores = normed @ query_normed
    ranked = np.argsort(scores)[::-1]
    results = []
    for i in ranked:
        slug = subject_slugs[i]
        if not any(ex in slug for ex in exclude):
            results.append((slug, float(scores[i])))
        if len(results) >= n:
            break
    return results


# Example: The Fool + The World - The Hermit ≈ ?
result_vec = vec("the-fool") + vec("the-world") - vec("the-hermit")
print("The Fool + The World - The Hermit:")
for slug, score in nearest(result_vec, n=5, exclude=["the-fool", "the-world", "the-hermit"]):
    print(f"  {score:.4f}  {slug}")